# TV-00d — Encodeur seul et encodeur-décodeur : les deux autres familles Transformer

Série **TransformerVariants**, née de l'epic [#16058](https://github.com/jsboige/CoursIA/issues/16058) ; ce carnet livre la ligne « encodeur seul et encodeur-décodeur from scratch » de l'EPIC [#18220](https://github.com/jsboige/CoursIA/issues/18220) (issue [#22699](https://github.com/jsboige/CoursIA/issues/22699)).

Le corpus de la série couvre le **décodeur seul** : l'architecture canonique dans
`3.4-Attention-Transformer-From-Scratch`, le positionnement RoPE dans
[TV-00a](TV-00a-RoPE-from-scratch.ipynb), les variantes d'attention MHA/MQA/GQA/SWA dans
[TV-00b](TV-00b-Attention-Variants-from-scratch.ipynb) (la mixture d'experts TV-00c est livrée
comme 3.4c dans la série 03-DeepLearning). Il manque les **deux autres familles** de
l'article original de 2017 : l'**encodeur seul** (bidirectionnel — la famille BERT) et
l'**encodeur-décodeur** (l'attention croisée — la famille T5). La différence entre les
trois familles n'est ni les paramètres ni la tête : c'est le **masque**.

**Ce que ce notebook mesure, et sur quoi il se prononce :**

| Question | Instrument |
|---|---|
| Les briques bidirectionnelles et croisées sont-elles correctes ? | échelle de vérifications : boucle par tête contre forme vectorisée, selfchecks de l'organe `tv` |
| Que gagne la bidirectionnalité sur la prédiction des trous ? | MLM sur un corpus français fermé : précision par taux de masquage (5/15/30 %), contre témoin causal à poids transférés puis ré-entraîné |
| L'attention croisée apprend-elle l'alignement source-cible ? | seq2seq de normalisation de dates : exact-match par pas d'entraînement, poids d'attention croisée lus position par position |
| Le décodage en faisceau change-t-il l'exact-match ? | glouton contre faisceau k=3 sur 500 dates de test |

Tout tourne sur CPU, en PyTorch seul : aucun `transformers`, aucun téléchargement.
Les briques d'attention viennent de l'organe `tv/model.py` (extrait de TV-00b, étendu
pour ce carnet) — l'attention multi-tête de base n'est pas re-dérivée ici.


## Sommaire

1. Trois familles, trois masques
2. Échelle de vérifications
3. Encodeur seul : prédire les trous (MLM)
4. Encodeur-décodeur : traduire des dates (seq2seq)
5. Les trois familles, côte à côte
6. Conclusion


In [1]:
import os
import sys
import math
import time

import numpy as np
import torch
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # determinisme cuBLAS (#16795)
# Determinisme (#16795) : la graine seule ne garantit PAS la reproductibilite
# (heuristiques cuDNN, kernels non deterministes). warn_only=True au premier
# passage pour inventorier les ops fautives sans faire echouer le run.
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, ".")
from tv import (
    Bloc,
    BlocDecodeur,
    PetitLM,
    PetitEncodeur,
    EncodeurDecodeur,
    AttnCroisee,
    attn_masked,
    attn_pleine,
    causal_window_mask,
    selfcheck_attn_equivalence,
    selfcheck_encodeurs,
)

DEVICE = "cpu"
GRAINE = 20261026
torch.manual_seed(GRAINE)
print("device :", DEVICE, "| torch :", torch.__version__, "| graine :", GRAINE)


device : cpu | torch : 2.13.0+cu126 | graine : 20261026


## 1. Trois familles, trois masques

Un bloc Transformer reçoit des triplets (Q, K, V) ; ce qui distingue les familles
est **qui a le droit de lire qui**. La brique `attn_masked` (masque causal) existait
depuis TV-00b ; ce carnet ajoute la brique manquante `attn_pleine` (aucun masque)
et les fait travailler ensemble :

- **Décodeur seul** (GPT, et toute la série TV jusqu'ici) : auto-attention
  **causale** — la requête `i` lit les positions `j <= i`. On peut générer, car
  aucun jeton ne dépend du futur.
- **Encodeur seul** (BERT) : auto-attention **bidirectionnelle** — `attn_pleine`
  sur le même tenseur. Chaque position lit toute la phrase, des deux côtés.
  On ne peut pas générer, mais chaque position est représentée en contexte
  complet : c'est la famille du *masked language modeling* (prédire les trous).
- **Encodeur-décodeur** (T5) : l'encodeur est bidirectionnel sur la source, le
  décodeur est causal sur la cible, et une **attention croisée** les relie —
  les requêtes viennent du décodeur, les clés et valeurs de la mémoire
  d'encodeur (`AttnCroisee`). C'est la famille de la traduction et, en
  général, de toute transduction séquence à séquence.

Le masque se matérialise en un booléen (T, T) pour les auto-attentions et
(T_dec, T_enc) pour la croisée. On les affiche : ligne = requête, colonne = clé.


In [2]:
def afficher_masque(m, etiquette):
    """Imprime un masque booleen : ligne = requete (Q), colonne = cle (K)."""
    print(f"{etiquette} ({tuple(m.shape)}) :")
    print("      " + " ".join(f"{j:2d}" for j in range(m.shape[-1])))
    for i in range(m.shape[0]):
        cases = " ".join(" ." if bool(m[i, j]) else " x" for j in range(m.shape[-1]))
        print(f"  Q{i:02d} {cases}")
    print()

T = 8
m_causal = causal_window_mask(T)               # decodeur seul : j <= i
m_plein = torch.ones(T, T, dtype=torch.bool)    # encodeur seul : tout le monde
T_dec = 5
m_croise = torch.ones(T_dec, T, dtype=torch.bool)  # croisee : toute la source

afficher_masque(m_causal[:6, :], "auto-attention CAUSALE (decodeur)")
afficher_masque(m_plein[:6, :], "auto-attention BIDIRECTIONNELLE (encodeur)")
afficher_masque(m_croise, "attention CROISEE (Q = decodeur, K = encodeur)")
print("lire : '.' = lecture autorisee, 'x' = interdite")


auto-attention CAUSALE (decodeur) ((6, 8)) :
       0  1  2  3  4  5  6  7
  Q00  .  x  x  x  x  x  x  x
  Q01  .  .  x  x  x  x  x  x
  Q02  .  .  .  x  x  x  x  x
  Q03  .  .  .  .  x  x  x  x
  Q04  .  .  .  .  .  x  x  x
  Q05  .  .  .  .  .  .  x  x

auto-attention BIDIRECTIONNELLE (encodeur) ((6, 8)) :
       0  1  2  3  4  5  6  7
  Q00  .  .  .  .  .  .  .  .
  Q01  .  .  .  .  .  .  .  .
  Q02  .  .  .  .  .  .  .  .
  Q03  .  .  .  .  .  .  .  .
  Q04  .  .  .  .  .  .  .  .
  Q05  .  .  .  .  .  .  .  .

attention CROISEE (Q = decodeur, K = encodeur) ((5, 8)) :
       0  1  2  3  4  5  6  7
  Q00  .  .  .  .  .  .  .  .
  Q01  .  .  .  .  .  .  .  .
  Q02  .  .  .  .  .  .  .  .
  Q03  .  .  .  .  .  .  .  .
  Q04  .  .  .  .  .  .  .  .

lire : '.' = lecture autorisee, 'x' = interdite


### 1.1 Lecture

Le masque causal est triangulaire inférieur — la diagonale incluse (un jeton
se lit lui-même). Le masque bidirectionnel est plein : c'est le causal **moins
la contrainte**, pas une autre formule. Le masque croisé est **rectangulaire** :
il ne protège aucun futur, parce que le « futur » qui compte (celui de la cible)
est déjà gardé par l'auto-attention causale du décodeur — la source, elle, est
entièrement connue au moment de générer chaque jeton. Les trois familles sont
donc trois politiques de partage d'information sur la **même** opération
`softmax(QK^T/sqrt(d))V`, ce que la section 2 vérifie avant toute mesure.


## 2. Échelle de vérifications

Avant de mesurer, on établit que les briques implémentent bien ce qu'elles
prétendent — contre des références **écrites différemment** :

- **(a)** `attn_pleine` contre une boucle par tête écrite à la main (softmax
  ligne par ligne) : même fonction, deux implémentations indépendantes ;
- **(b)** `AttnCroisee` contre la même boucle par tête, requêtes et clés de
  provenances différentes ;
- **(c)** selfchecks de l'organe (`tv.model.selfcheck_encodeurs`) : le bloc
  bidirectionnel **diffère** du bloc causal à poids égaux, et l'attention
  croisée est invariante à la permutation des requêtes ;
- **(d)** non-régression : le selfcheck historique de TV-00b (bande contre
  masque) passe toujours.


In [3]:
def boucle_par_tete(q, k, v):
    """Reference naive : une tete, une position, une boucle. Lent, lisible, independant."""
    B, H, T, dh = q.shape
    S = k.shape[2]
    out = torch.zeros_like(q)
    for b in range(B):
        for h in range(H):
            scores = torch.zeros(T, S)
            for i in range(T):
                for j in range(S):
                    scores[i, j] = torch.dot(q[b, h, i], k[b, h, j]) / math.sqrt(dh)
            poids = torch.softmax(scores, dim=-1)
            for i in range(T):
                acc = torch.zeros(dh)
                for j in range(S):
                    acc += poids[i, j] * v[b, h, j]
                out[b, h, i] = acc
    return out


g = torch.Generator().manual_seed(11)
B, H, T, dh = 2, 3, 6, 8
q = torch.randn(B, H, T, dh, generator=g)
k = torch.randn(B, H, T, dh, generator=g)
v = torch.randn(B, H, T, dh, generator=g)

# (a) attention pleine contre boucle par tete
with torch.no_grad():
    e_a = float((attn_pleine(q, k, v) - boucle_par_tete(q, k, v)).abs().max())
print(f"(a) attn_pleine vs boucle par tete      : ecart max {e_a:.2e}")

# (b) attention croisee contre boucle par tete (T_dec != T_enc)
T_dec_ex = 4
croisee = AttnCroisee(d_model=H * dh, n_heads=H)
x_dec = torch.randn(B, T_dec_ex, H * dh, generator=g)
mem_enc = torch.randn(B, T, H * dh, generator=g)
with torch.no_grad():
    o_module = croisee(x_dec, mem_enc)
    qp = croisee.q_proj(x_dec).view(B, T_dec_ex, H, dh).transpose(1, 2)
    kp = croisee.k_proj(mem_enc).view(B, T, H, dh).transpose(1, 2)
    vp = croisee.v_proj(mem_enc).view(B, T, H, dh).transpose(1, 2)
    o_ref = boucle_par_tete(qp, kp, vp)
    o_ref_proj = croisee.o_proj(o_ref.transpose(1, 2).reshape(B, T_dec_ex, H * dh))
    e_b = float((o_module - o_ref_proj).abs().max())
print(f"(b) AttnCroisee vs boucle par tete      : ecart max {e_b:.2e}")

# (c) selfchecks de l'organe (bidirectionnel != causal, invariance croisee)
sc = selfcheck_encodeurs()
print(f"(c) selfcheck_encodeurs                 : passe = {sc['passed']} "
      f"(bi vs causal pos 0 : {sc['a_ecart_bi_vs_causal_pos0']:.2e} ; "
      f"permutation decodeur : {sc['b_ecart_permutation_decodeur']:.2e})")

# (d) non-regression TV-00b
sc_b = selfcheck_attn_equivalence()
print(f"(d) selfcheck TV-00b (bande vs masque)  : passe = {sc_b['passed']} "
      f"(ecart max {sc_b['max_abs_diff']:.2e})")


(a) attn_pleine vs boucle par tete      : ecart max 1.19e-07
(b) AttnCroisee vs boucle par tete      : ecart max 5.96e-08
(c) selfcheck_encodeurs                 : passe = True (bi vs causal pos 0 : 1.30e+00 ; permutation decodeur : 0.00e+00)
(d) selfcheck TV-00b (bande vs masque)  : passe = True (ecart max 1.79e-07)


### 2.1 Lecture

Les quatre vérifications rendent le même verdict : les briques vectorisées
calculent la même fonction que les boucles de référence à la précision float32
près, l'organe historique n'a pas bougé, et le selfcheck (c) confirme les deux
invariants propres aux nouvelles familles — **retirer le masque causal change
la sortie** (l'encodeur n'est pas un décodeur déguisé), et **permuter les
requêtes de l'attention croisée** les laisse inchangées (chaque position du
décodeur est traitée indépendamment, avant que l'auto-attention causale ne les
mélange). On peut maintenant mesurer.


## 3. Encodeur seul : prédire les trous (MLM)

Le *masked language modeling* (Devlin et al., 2018 — BERT) masque une fraction
des jetons et demande au modèle de les prédire **à partir du reste**, des deux
côtés. C'est la tâche qui justifie la bidirectionnalité : un décodeur causal
ne voit que le contexte gauche.

**Corpus.** Un micro-langage français fermé, généré par gabarits — sujet,
verbe, objet avec adjectif, complément de lieu — dix mots par phrase :

> `le chat fidèle miaule la souris jeune dans le jardin`

Deux dépendances traversent la phrase : la **collocation sujet-verbe**
(`miaule` n'appartient qu'au chat) et l'**accord du déterminant** (`le/la`
porté par le nom qui le suit). Le verbe (position 3) se prédit depuis le
sujet (position 1, à sa **gauche**) ; mais réciproquement le **sujet** ne se
prédit bien que depuis le **verbe qui le suit** — et `le/la` que depuis le
**nom qui suit**. Ces positions-là sont le terrain de la bidirectionnalité ;
le verbe, lui, servira de **témoin gauche** : une dépendance que le masque
causal lit au moins aussi bien — la mesure en 3.4 montrera qu'il la lit
même **mieux**.

**Masquage 80/10/10** (la recette BERT) : parmi les positions tirées au
masque, 80 % reçoivent le jeton `[MASK]`, 10 % un jeton aléatoire du
vocabulaire, 10 % restent inchangées. La perte ne compte que les positions
sélectionnées.


In [4]:
# --- corpus ferme par gabarits -------------------------------------------
NOMS = ["chat", "chienne", "renard", "souris", "lapine", "oiseau"]
GENRE = {"chat": "m", "chienne": "f", "renard": "m", "souris": "f", "lapine": "f", "oiseau": "m"}
# collocation sujet-verbe : le verbe n'appartient qu'a un seul nom
VERBE_PAR_NOM = {"chat": "miaule", "chienne": "aboie", "renard": "ruse",
                 "souris": "grignotte", "lapine": "bondit", "oiseau": "chante"}
ADJS = ["rapide", "jeune", "fidèle", "agile", "tranquille", "farouche"]  # invariables en genre
PREPS = ["dans", "sous", "vers"]
LIEUX = ["jardin", "arbre", "grenier", "nichoir", "buisson"]  # tous masculins -> "le"


def phrase_aleatoire(g):
    """Une phrase du micro-langage, collocation sujet-verbe + accords le/la."""
    def det(nom):
        return "le" if GENRE[nom] == "m" else "la"
    s, o = (NOMS[torch.randint(0, len(NOMS), (1,), generator=g).item()] for _ in range(2))
    a1, a2 = (ADJS[torch.randint(0, len(ADJS), (1,), generator=g).item()] for _ in range(2))
    p = PREPS[torch.randint(0, len(PREPS), (1,), generator=g).item()]
    l = LIEUX[torch.randint(0, len(LIEUX), (1,), generator=g).item()]
    return [det(s), s, a1, VERBE_PAR_NOM[s], det(o), o, a2, p, "le", l]


MOTS = sorted({w for d in (NOMS, ADJS, PREPS, LIEUX) for w in d}
              | set(VERBE_PAR_NOM.values()) | {"le", "la"})
VOCAB = MOTS + ["[MASK]", "[PAD]"]
TOK = {w: i for i, w in enumerate(VOCAB)}
ID_MASQUE, ID_PAD = TOK["[MASK]"], TOK["[PAD]"]
# positions sondées : le sujet (depend du verbe, à sa DROITE) et le verbe (depend du sujet, à GAUCHE)
T_MLM, POS_SUJET, POS_VERBE = 10, 1, 3


def encoder_phrases(phrases):
    ids = torch.full((len(phrases), T_MLM), ID_PAD, dtype=torch.long)
    for i, p in enumerate(phrases):
        ids[i, :len(p)] = torch.tensor([TOK[w] for w in p])
    return ids


def appliquer_masquage(ids, taux, g):
    """Recette 80/10/10 : retourne (ids_masques, positions_selectionnees)."""
    sel = (torch.rand(ids.shape, generator=g) < taux) & (ids != ID_PAD)
    ids_m = ids.clone()
    tirage = torch.rand(ids.shape, generator=g)
    aleatoires = sel & (tirage < 0.10)          # 10 % -> jeton aleatoire
    inchanges = sel & (tirage >= 0.90)          # 10 % -> inchanges
    masques = sel & ~aleatoires & ~inchanges    # 80 % -> [MASK]
    mots_aleatoires = torch.randint(0, len(MOTS), ids.shape, generator=g)
    ids_m[masques] = ID_MASQUE
    ids_m[aleatoires] = mots_aleatoires[aleatoires]
    return ids_m, sel


torch.manual_seed(GRAINE)
g_data = torch.Generator().manual_seed(GRAINE)
train_phr = [phrase_aleatoire(g_data) for _ in range(4096)]
g_test = torch.Generator().manual_seed(555)  # genere une seule fois : 512 phrases distinctes
test_phr = [phrase_aleatoire(g_test) for _ in range(512)]
assert len({" ".join(p) for p in test_phr}) > 400, "le test doit contenir des phrases distinctes"
print("vocabulaire :", len(VOCAB), "jetons | longueur phrase :", T_MLM, "| exemple :", " ".join(train_phr[0]))
ids_ex, sel_ex = appliquer_masquage(encoder_phrases(train_phr[:4]), 0.15, g_data)
for i in range(2):
    montre = [VOCAB[t] if t != ID_PAD else "_" for t in ids_ex[i].tolist()]
    print("masquee  :", " ".join(montre), "| selectionnees :", int(sel_ex[i].sum()))


vocabulaire : 30 jetons | longueur phrase : 10 | exemple : le oiseau fidèle chante le oiseau fidèle sous le buisson
masquee  : le oiseau fidèle chante le oiseau fidèle sous le buisson | selectionnees : 1
masquee  : la [MASK] rapide [MASK] la lapine farouche sous le jardin | selectionnees : 2


### 3.1 Entraînement

`PetitEncodeur` (organe `tv`) : plongements + 2 blocs bidirectionnels +
normalisation + tête, une prédiction **par position**. La perte ne compte que
les positions sélectionnées par le masquage — toutes les positions
participent à la représentation, seules les sélectionnées sont supervisées.


In [5]:
def perte_mlm(logits, cibles, sel):
    """Entropie croisee moyenne sur les seules positions selectionnees."""
    return F.cross_entropy(logits[sel], cibles[sel])


def entrainer_mlm(modele, phrases, taux, pas, taille_lot=64, lr=3e-3, g=None, journal=50):
    opt = torch.optim.Adam(modele.parameters(), lr=lr)
    historique = []
    for etape in range(1, pas + 1):
        i = torch.randint(0, len(phrases) - taille_lot, (1,), generator=g).item()
        ids = encoder_phrases(phrases[i:i + taille_lot])
        ids_m, sel = appliquer_masquage(ids, taux, g)
        perte = perte_mlm(modele(ids_m), ids, sel)
        opt.zero_grad(); perte.backward(); opt.step()
        if etape % journal == 0 or etape == 1:
            historique.append((etape, perte.item()))
    return historique


def precision_mlm(modele, phrases, taux, g, position=None):
    """Precision top-1 sur les positions selectionnees ; position=None -> toutes,
    position=k -> seulement la position k du gabarit (ex. le verbe)."""
    ids = encoder_phrases(phrases)
    ids_m, sel = appliquer_masquage(ids, taux, g)
    if position is not None:
        sel = sel.clone()
        sel[:, :position] = False
        sel[:, position + 1:] = False
    with torch.no_grad():
        pred = modele(ids_m).argmax(-1)
    return int(((pred == ids) & sel).sum()) / max(int(sel.sum()), 1)


D_MODEL, N_TETES, N_COUCHES, TAUX = 64, 4, 2, 0.15
torch.manual_seed(GRAINE)
enc_bi = PetitEncodeur(vocab=len(VOCAB), d_model=D_MODEL, n_heads=N_TETES, n_couches=N_COUCHES)
print(f"PetitEncodeur : {enc_bi.n_parametres():,} parametres "
      f"(d={D_MODEL}, tetes={N_TETES}, couches={N_COUCHES})")
t0 = time.time()
hist_bi = entrainer_mlm(enc_bi, train_phr, TAUX, pas=600, journal=100,
                        g=torch.Generator().manual_seed(GRAINE))
print(f"600 pas en {time.time() - t0:.1f} s | perte : "
      + " -> ".join(f"{p:.3f} (pas {e})" for e, p in hist_bi))


PetitEncodeur : 103,424 parametres (d=64, tetes=4, couches=2)


600 pas en 20.5 s | perte : 3.422 (pas 1) -> 1.950 (pas 100) -> 1.494 (pas 200) -> 1.539 (pas 300) -> 1.728 (pas 400) -> 1.650 (pas 500) -> 1.206 (pas 600)


### 3.2 Lecture

La perte s'effondre puis se stabilise : sur un langage fermé, le plafond
n'est pas la capacité du modèle mais l'**aléa du gabarit** — les adjectifs,
prépositions et lieux sont tirés uniformément, aucune prédiction ne peut
dépasser leur fréquence a priori. Les positions **liées** (déterminants,
sujet, verbe) peuvent en revanche monter bien plus haut : c'est l'écart
entre les deux régimes que la mesure suivante sépare.


In [6]:
# --- mesure clee 1 : precision par taux de masquage ----------------------
g_eval = torch.Generator().manual_seed(99)
print("precision top-1 sur positions selectionnees (512 phrases de test)")
print(f"{'taux':>6} | {'globale':>8} | {'sujet (lit a droite)':>20} | {'verbe (lit a gauche)':>21}")
for taux in (0.05, 0.15, 0.30):
    glob = precision_mlm(enc_bi, test_phr, taux, g_eval)
    suj = precision_mlm(enc_bi, test_phr, taux, g_eval, position=POS_SUJET)
    verb = precision_mlm(enc_bi, test_phr, taux, g_eval, position=POS_VERBE)
    print(f"{taux:>6.0%} | {glob:>8.1%} | {suj:>20.1%} | {verb:>21.1%}")

# --- lecture des predictions ---------------------------------------------
ids = encoder_phrases(test_phr[:3])
ids_m, sel = appliquer_masquage(ids, 0.15, torch.Generator().manual_seed(99))
with torch.no_grad():
    pred = enc_bi(ids_m).argmax(-1)
print()
for i in range(3):
    ligne = []
    for j in range(T_MLM):
        if ids[i, j] == ID_PAD:
            continue
        if sel[i, j]:
            marque = "+" if pred[i, j] == ids[i, j] else "-"
            ligne.append(f"[{VOCAB[pred[i, j]]}]{marque}")
        else:
            ligne.append(VOCAB[ids[i, j]])
    print(" ".join(ligne), "  (+ = trou bien predit)")


precision top-1 sur positions selectionnees (512 phrases de test)
  taux |  globale | sujet (lit a droite) |  verbe (lit a gauche)
    5% |    53.6% |                83.3% |                 54.8%


   15% |    46.6% |                58.2% |                 41.9%


   30% |    36.9% |                51.3% |                 20.1%

[la]+ lapine farouche bondit la lapine [la]- vers le nichoir   (+ = trou bien predit)
le chat jeune miaule la lapine agile sous le buisson   (+ = trou bien predit)
[la]+ [la]- rapide bondit le renard tranquille vers le buisson   (+ = trou bien predit)


### 3.3 Lecture — le taux de masquage n'est pas neutre

Les deux colonnes sondées décroissent quand le taux monte — c'est la
lecture standard du choix de BERT (15 %) : à 30 %, le contexte se raréfie.
Mais le classement des deux positions surprend : le **sujet**, dont la
preuve forte (le verbe) est à sa **droite**, garde la meilleure précision à
tous les taux, devant le **verbe** dont la preuve (le nom sujet) est
pourtant à gauche. La raison est la **redondance** : le sujet a deux
preuves (le déterminant `la/le` qui accorde le genre juste à gauche, le
verbe qui identifie le nom à droite) quand le verbe n'en a qu'une — quand
le taux monte, une seule preuve masquée suffit à casser la position.

Cette lecture laisse une question ouverte : le sujet lit-il vraiment à
droite, ou le genre du déterminant suffit-il ? Le corps du carnet ne peut
pas trancher à ce stade — c'est exactement ce que le témoin suivant isole,
en changeant **une seule chose** : le masque.


In [7]:
# --- temoin negatif 1 : les memes poids, le masque causal -----------------
enc_transfert = PetitEncodeur(vocab=len(VOCAB), d_model=D_MODEL, n_heads=N_TETES, n_couches=N_COUCHES)
enc_transfert.load_state_dict(enc_bi.state_dict())  # poids ENTRAINES du bidirectionnel
for b in enc_transfert.blocs:
    b.attn.bidirectionnel = False                   # seul changement : le masque

# --- temoin negatif 2 : causal entraine from scratch, meme budget ----------
torch.manual_seed(GRAINE)
enc_causal = PetitEncodeur(vocab=len(VOCAB), d_model=D_MODEL, n_heads=N_TETES, n_couches=N_COUCHES)
for b in enc_causal.blocs:
    b.attn.bidirectionnel = False
hist_c = entrainer_mlm(enc_causal, train_phr, TAUX, pas=600, journal=100,
                       g=torch.Generator().manual_seed(GRAINE))
print(f"causal re-entraine (600 pas) : perte finale {hist_c[-1][1]:.3f} "
      f"(bidirectionnel : {hist_bi[-1][1]:.3f})")
print()
print(f"{'taux':>6} | {'bi':>6} | {'transfert causal':>17} | {'causal re-entraine':>19} "
      f"| {'sujet bi':>9} | {'sujet causal':>13} | {'verbe bi':>9} | {'verbe causal':>13}")
for taux in (0.05, 0.15, 0.30):
    g_ligne = torch.Generator().manual_seed(99)
    gb = precision_mlm(enc_bi, test_phr, taux, g_ligne)
    gt = precision_mlm(enc_transfert, test_phr, taux, g_ligne)
    gc = precision_mlm(enc_causal, test_phr, taux, g_ligne)
    sb = precision_mlm(enc_bi, test_phr, taux, g_ligne, position=POS_SUJET)
    sc = precision_mlm(enc_causal, test_phr, taux, g_ligne, position=POS_SUJET)
    vb = precision_mlm(enc_bi, test_phr, taux, g_ligne, position=POS_VERBE)
    vc = precision_mlm(enc_causal, test_phr, taux, g_ligne, position=POS_VERBE)
    print(f"{taux:>6.0%} | {gb:>6.1%} | {gt:>17.1%} | {gc:>19.1%} | {sb:>9.1%} | {sc:>13.1%} "
          f"| {vb:>9.1%} | {vc:>13.1%}")


causal re-entraine (600 pas) : perte finale 1.168 (bidirectionnel : 1.206)

  taux |     bi |  transfert causal |  causal re-entraine |  sujet bi |  sujet causal |  verbe bi |  verbe causal


    5% |  53.6% |             28.9% |               50.8% |     76.9% |         56.0% |     65.0% |         90.0%


   15% |  43.1% |             24.7% |               46.9% |     63.7% |         31.6% |     41.9% |         90.0%
   30% |  34.6% |             21.4% |               41.0% |     43.0% |         24.1% |     25.7% |         74.5%


### 3.4 Lecture — le masque EST la famille

Deux témoins, trois lectures distinctes :

1. **Poids transférés** : le modèle bidirectionnel entraîné, privé de son
   contexte droit par simple bascule du masque, s'effondre à tous les taux
   (53.6 → 28.9 %, 43.1 → 24.7 %, 34.6 → 21.4 %) — sans réapprendre. La
   représentation qu'il avait construite **suppose** de lire les deux
   côtés : couper la lecture droite après coup casse les prédictions qui
   en dépendaient. Le masque n'est pas un réglage post-hoc.
2. **Entraîné causal dès le départ, même budget** : la colonne « globale »
   reste au coude-à-coude (le causal repasse même devant à 15 et 30 %) —
   normal : sur ce gabarit, la plupart des positions ont leur preuve à
   gauche ou n'en ont pas. L'écart net est sur la colonne **sujet** :
   76.9 contre 56.0 % à 5 %, 63.7 contre 31.6 % à 15 %, 43.0 contre
   24.1 % à 30 % — la position dont la preuve forte (le verbe) est à
   droite, et que 3.3 ne pouvait pas isoler du genre du déterminant.
3. **Le contrôle symétrique** : les colonnes « verbe » mesurent la
   dépendance inverse (preuve à **gauche**) — et l'avantage s'y inverse :
   90.0 contre 65.0 % à 5 %, 90.0 contre 41.9 % à 15 %, 74.5 contre
   25.7 % à 30 %. La fenêtre de lecture coupe **dans les deux sens** :
   le causal concentre son attention sur les positions qui portent
   l'information, quand le bidirectionnel la dilue aussi sur une moitié
   droite qui, pour le verbe, ne porte rien. La bidirectionnalité n'est
   pas un sur-ensemble gratuit du causal : c'est un **échange** — la
   lecture droite gagnée sur le sujet, la focalisation gauche perdue sur
   le verbe.

C'est la démonstration minimale du choix de BERT : bidirectionnalité =
information droite exploitable — visible **position par position**, pas
dans une moyenne globale — payée par l'impossibilité de générer.


## Exercice 1 — écrire le masquage 80/10/10

**Objectif.** Réécrire le cœur du masquage : la fonction `repartition_801010`
ci-dessous reçoit le tenseur `sel` des positions sélectionnées et le tenseur
`tirage` des valeurs uniformes, et retourne le triplet de masques booléens
`(masque, aleatoire, inchange)` — 80 % / 10 % / 10 % **parmi les
sélectionnées**, et rien du tout ailleurs.


In [8]:
def repartition_801010(sel, tirage):
    """Exercice 1 : trois masques disjoints, union = sel, proportions 80/10/10."""
    # TODO etudiant : seuiller tirage en trois zones, puis intersecter avec sel
    resultat = None
    return resultat


# verification (a completer apres l'exercice) :
sel_t = torch.zeros(2, 10, dtype=torch.bool)
sel_t[:, 2:12:2] = True  # 5 positions selectionnees par ligne
tirage_t = torch.rand(2, 10, generator=torch.Generator().manual_seed(3))
rep = repartition_801010(sel_t, tirage_t)
if rep is None:
    print("Exercice 1 a completer : repartition_801010 a renvoye None")
else:
    masque, aleatoire, inchange = rep
    union_ok = bool((masque | aleatoire | inchange).equal(sel_t))
    disjoint = bool(not (masque & aleatoire).any() and not (masque & inchange).any()
                    and not (aleatoire & inchange).any())
    hors_sel = bool(not (masque & ~sel_t).any() and not (aleatoire & ~sel_t).any()
                    and not (inchange & ~sel_t).any())
    print("union = sel :", union_ok, "| disjoints :", disjoint, "| rien hors sel :", hors_sel)
    print("effectifs ligne 0 : masque", int(masque[0].sum()), "| aleatoire", int(aleatoire[0].sum()),
          "| inchange", int(inchange[0].sum()))


Exercice 1 a completer : repartition_801010 a renvoye None


## 4. Encodeur-décodeur : traduire des dates (seq2seq)

La troisième famille transduit une séquence en une autre. Tâche jouet mais
complète : **normaliser une date française en ISO** — et pas simplement la
recopier : la source présente le jour et le mois sous des formes de surface
**variables** (mois en toutes lettres ou en chiffres, jour avec ou sans zéro
de tête), que la cible canonise :

> source : `le 3 avril <annee> 1 8 9 7` → cible : `1 8 9 7 - 04 - 03`
> source : `le 03 04 <annee> 1 8 9 7` → cible : `1 8 9 7 - 04 - 03`

Le vocabulaire est minuscule (jours, douze mois, chiffres, séparateurs), la
cible est **exactement** vérifiable (exact-match), et l'alignement
source-cible est non trivial : l'année en sortie se lit en **fin** de
source, le jour en **début** — le modèle doit apprendre à aller chercher
l'information au bon endroit, et les poids de l'attention croisée
montreront **où** il est allé la chercher. C'est le même rôle que la
traduction dans l'article de 2017, à une échelle où chaque position reste
lisible à l'œil.


In [9]:
MOIS = ["janvier", "février", "mars", "avril", "mai", "juin", "juillet",
         "août", "septembre", "octobre", "novembre", "décembre"]
JOURS_PAR_MOIS = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]  # février toujours 28 (simplification assumée)

VOCAB_S2S = ["<pad>", "<bos>", "<eos>", "le", "<annee>"] \
    + [str(j) for j in range(1, 32)] + MOIS \
    + [c for c in "0123456789"] + [f"{m:02d}" for m in range(1, 13)] \
    + [f"{d:02d}" for d in range(1, 32)] + ["-"]
VOCAB_S2S = list(dict.fromkeys(VOCAB_S2S))  # dédoublonne en gardant l'ordre
TOK_S = {w: i for i, w in enumerate(VOCAB_S2S)}
ID_PAD_S, ID_BOS, ID_EOS = 0, 1, 2


def paire_date(g):
    """(source, cible) : formes de surface variables -> ISO canonique.

    Le mois arrive en lettres ou en chiffres, le jour avec ou sans zéro de
    tête (p = 0.5 chacun) ; la cible est toujours 'AAAA - MM - JJ'.
    """
    m = torch.randint(0, 12, (1,), generator=g).item()
    j = torch.randint(0, JOURS_PAR_MOIS[m], (1,), generator=g).item()
    a = 1800 + torch.randint(0, 200, (1,), generator=g).item()
    mois_src = MOIS[m] if torch.rand(1, generator=g).item() < 0.5 else f"{m + 1:02d}"
    jour_src = str(j + 1) if torch.rand(1, generator=g).item() < 0.5 else f"{j + 1:02d}"
    src = ["le", jour_src, mois_src, "<annee>"] + list(str(a))
    tgt = list(str(a)) + ["-", f"{m + 1:02d}", "-", f"{j + 1:02d}"]
    return src, tgt


T_ENC, T_DEC = 8, 10  # src : le + jour + mois + <annee> + 4 chiffres = 8 ; tgt max : 4 + 4 = 8 jetons


def encoder_s2s(paires):
    """src pad a droite ; tgt = <bos> + cible ; out = cible + <eos> (cible du dernier pas)."""
    B = len(paires)
    src = torch.full((B, T_ENC), ID_PAD_S, dtype=torch.long)
    tgt = torch.full((B, T_DEC), ID_PAD_S, dtype=torch.long)
    out = torch.full((B, T_DEC), ID_PAD_S, dtype=torch.long)
    for i, (s, t) in enumerate(paires):
        src[i, :len(s)] = torch.tensor([TOK_S[w] for w in s])
        tgt[i, 0] = ID_BOS
        tgt[i, 1:1 + len(t)] = torch.tensor([TOK_S[w] for w in t])
        out[i, :len(t)] = torch.tensor([TOK_S[w] for w in t])
        out[i, len(t)] = ID_EOS
    return src, tgt, out


torch.manual_seed(GRAINE)
g_s2s = torch.Generator().manual_seed(GRAINE)
train_dates = [paire_date(g_s2s) for _ in range(4096)]
g_test_d = torch.Generator().manual_seed(555)  # genere une seule fois : 500 dates distinctes
test_dates = [paire_date(g_test_d) for _ in range(500)]
assert len({" ".join(s) for s, _ in test_dates}) > 400, "le test doit contenir des dates distinctes"
print("vocabulaire source + cible :", len(VOCAB_S2S), "jetons | src :", T_ENC, "pos | cible <= ", T_DEC, "pos")
for s, t in train_dates[:3]:
    print("src :", " ".join(s), "-> tgt :", " ".join(t))


vocabulaire source + cible : 59 jetons | src : 8 pos | cible <=  10 pos
src : le 06 décembre <annee> 1 8 9 0 -> tgt : 1 8 9 0 - 12 - 06
src : le 09 05 <annee> 1 9 6 2 -> tgt : 1 9 6 2 - 05 - 09
src : le 07 août <annee> 1 8 5 6 -> tgt : 1 8 5 6 - 08 - 07


In [10]:
def chaine(tokens):
    mots = []
    for t in tokens.tolist():
        if t in (ID_EOS, ID_PAD_S):
            break
        mots.append(VOCAB_S2S[t])
    return " ".join(mots)


def decoder_glouton(modele, src, max_pas=T_DEC):
    """Decodage pas a pas : a chaque pas, le jeton le plus probable."""
    mem = modele.memoire(src)
    prefixes = torch.full((src.shape[0], 1), ID_BOS, dtype=torch.long)
    finis = torch.zeros(src.shape[0], dtype=torch.bool)
    for _ in range(max_pas):
        with torch.no_grad():
            logits = modele.pas_decodage(prefixes, mem)
        suiv = logits.argmax(-1)
        suiv[finis] = ID_EOS
        prefixes = torch.cat([prefixes, suiv.unsqueeze(1)], dim=1)
        finis = finis | (suiv == ID_EOS)
        if bool(finis.all()):
            break
    return prefixes[:, 1:]


def decoder_faisceau(modele, src, k=3, max_pas=T_DEC):
    """Faisceau k : les k prefixes les plus vraisemblables en score cumule.

    Un candidat termine (EOS emis) continue de concourir au meme score :
    sa distribution est forcee a EOS seul, a cout nul. Le jeton <pad> est
    exclu partout.
    """
    B, V = src.shape[0], len(VOCAB_S2S)
    mem = modele.memoire(src)
    prefixes = torch.full((B, 1, 1), ID_BOS, dtype=torch.long)   # (B, k_cur, T)
    scores = torch.zeros(B, 1)
    finis = torch.zeros(B, 1, dtype=torch.bool)
    for etape in range(max_pas):
        kc = prefixes.shape[1]
        aplat = prefixes.reshape(B * kc, -1)
        mem_et = mem.unsqueeze(1).expand(-1, kc, -1, -1).reshape(B * kc, mem.shape[1], -1)
        with torch.no_grad():
            logits = modele.pas_decodage(aplat, mem_et)           # (B*kc, V)
        logp = torch.log_softmax(logits, -1).view(B, kc, V)
        logp[:, :, ID_PAD_S] = float("-inf")
        logp = logp.masked_fill(finis.unsqueeze(-1), float("-inf"))
        logp[:, :, ID_EOS] = torch.where(finis, torch.zeros_like(logp[:, :, ID_EOS]),
                                         logp[:, :, ID_EOS])
        if etape == 0:
            logp[:, 1:, :] = float("-inf")                        # un seul candidat au depart
        cumules = (scores.unsqueeze(-1) + logp).reshape(B, -1)
        top_scores, top_idx = cumules.topk(min(k, cumules.shape[1]), dim=-1)
        scores = top_scores
        candidat = torch.div(top_idx, V, rounding_mode="floor")
        jeton = top_idx % V
        bases = torch.gather(prefixes, 1, candidat.unsqueeze(-1).expand(-1, -1, prefixes.shape[-1]))
        prefixes = torch.cat([bases, jeton.unsqueeze(-1)], dim=-1)
        finis = torch.gather(finis, 1, candidat) | (jeton == ID_EOS)
        if bool(finis.all()):
            break
    meilleur = scores.argmax(-1)
    return torch.stack([prefixes[b, meilleur[b], 1:] for b in range(B)])


def exact_match(modele, paires, faisceau=False, k=3):
    src, tgt, out = encoder_s2s(paires)
    pred = decoder_faisceau(modele, src, k=k) if faisceau else decoder_glouton(modele, src)
    ok, details = 0, []
    for i, (s, t) in enumerate(paires):
        p = chaine(pred[i])
        c = " ".join(t)
        ok += int(p == c)
        details.append((p == c, " ".join(s), c, p))
    return ok / len(paires), details


# --- mesure clee 2 : exact-match au fil de l'entrainement -----------------
D_S2S = 64
torch.manual_seed(GRAINE)
s2s = EncodeurDecodeur(vocab=len(VOCAB_S2S), d_model=D_S2S, n_heads=N_TETES, n_couches=N_COUCHES)
print(f"EncodeurDecodeur : {s2s.n_parametres():,} parametres")

PAS_TOTAL, TAILLE_LOT, LR = 400, 64, 2e-3
opt = torch.optim.Adam(s2s.parameters(), lr=LR)
g_entr = torch.Generator().manual_seed(GRAINE)
t0 = time.time()
courbe = []  # (etape, exact-match glouton, exact-match faisceau k=3)
print("exact-match sur 500 dates de test distinctes, au fil de l'entraînement :")
print(f"{'pas':>5} | {'perte':>6} | {'glouton':>8} | {'faisceau k=3':>13}")
for etape in range(1, PAS_TOTAL + 1):
    i = torch.randint(0, len(train_dates) - TAILLE_LOT, (1,), generator=g_entr).item()
    src, tgt, out = encoder_s2s(train_dates[i:i + TAILLE_LOT])
    logits = s2s(src, tgt)
    perte = F.cross_entropy(logits.reshape(-1, len(VOCAB_S2S)), out.reshape(-1),
                            ignore_index=ID_PAD_S)
    opt.zero_grad(); perte.backward()
    torch.nn.utils.clip_grad_norm_(s2s.parameters(), 1.0)
    opt.step()
    if etape % 50 == 0:
        em_g, _ = exact_match(s2s, test_dates)
        em_f, _ = exact_match(s2s, test_dates, faisceau=True)
        courbe.append((etape, em_g, em_f))
        print(f"{etape:>5} | {perte.item():>6.3f} | {em_g:>8.1%} | {em_f:>13.1%}")
print(f"entraînement + évaluations en {time.time() - t0:.1f} s")


EncodeurDecodeur : 239,744 parametres
exact-match sur 500 dates de test distinctes, au fil de l'entraînement :
  pas |  perte |  glouton |  faisceau k=3


   50 |  0.749 |     2.0% |          2.8%


  100 |  0.283 |    36.2% |         34.2%


  150 |  0.231 |    37.0% |         36.6%


  200 |  0.164 |    38.6% |         39.0%


  250 |  0.133 |    37.0% |         36.8%


  300 |  0.128 |    38.0% |         38.0%


  350 |  0.139 |    38.8% |         39.0%


  400 |  0.147 |    43.2% |         42.0%
entraînement + évaluations en 29.5 s


### 4.1 Lecture

L'exact-match est la métrique sans pitié du seq2seq : une date normalisée
est bonne ou fausse, chiffre par chiffre — 95 % de jetons corrects peut
faire 0 % de dates correctes. La montée se lit en deux temps : la perte
s'effondre dès le pas 100 (0.75 → 0.28) pendant que l'exact-match décolle
(2 % → 36 %), puis les deux continuent de s'améliorer lentement jusqu'à
~43 % — copier l'année est appris vite, départager le jour du mois quand
les deux se présentent en chiffres (`le 8 02`) est le conflit qui reste.
La colonne faisceau colle au glouton à ±2 points sur toute la courbe :
aucun des deux décodages ne domine l'autre ici, et la section 4.2
explique pourquoi avec les exemples dans les deux sens.


In [11]:
# --- mesure clee 3 : lecture du contraste glouton / faisceau -------------
etape_f, em_g_f, em_f_f = courbe[-1]
print(f"modele final (pas {etape_f}) : glouton {em_g_f:.1%} | faisceau k=3 {em_f_f:.1%} "
      f"| delta {em_f_f - em_g_f:+.1%}")
# premiere etape evaluee ou le glouton est encore imparfait : le contraste s'y lit mieux
partiel = next(((e, g, f) for e, g, f in courbe if 0.05 < g < 0.95), None)
if partiel:
    e_p, g_p, f_p = partiel
    print(f"convergence partielle (pas {e_p}) : glouton {g_p:.1%} | faisceau k=3 {f_p:.1%} "
          f"| delta {f_p - g_p:+.1%}")

em_g, details_g = exact_match(s2s, test_dates)
em_f, details_f = exact_match(s2s, test_dates, faisceau=True)
pred_faisceau = {s: p for v, s, c, p in details_f}
gagnes = [x for x in details_f if x[0] and not next(y[0] for y in details_g if y[1] == x[1])]
for v, s, c, p in gagnes[:3]:
    print(f"  faisceau gagne : {s} -> attendu {c!r}")
perdus = [x for x in details_g if x[0] and not next(y[0] for y in details_f if y[1] == x[1])]
for v, s, c, p in perdus[:3]:
    print(f"  faisceau perd  : {s} -> faisceau a produit {pred_faisceau[s]!r}")


modele final (pas 400) : glouton 43.2% | faisceau k=3 42.0% | delta -1.2%
convergence partielle (pas 100) : glouton 36.2% | faisceau k=3 34.2% | delta -2.0%


  faisceau gagne : le 06 juin <annee> 1 9 4 9 -> attendu '1 9 4 9 - 06 - 06'
  faisceau gagne : le 8 02 <annee> 1 8 5 1 -> attendu '1 8 5 1 - 02 - 08'
  faisceau gagne : le 12 11 <annee> 1 8 7 2 -> attendu '1 8 7 2 - 11 - 12'
  faisceau perd  : le 20 08 <annee> 1 8 6 2 -> faisceau a produit '1 8 2 6 - 08 - 20'
  faisceau perd  : le 23 novembre <annee> 1 8 9 8 -> faisceau a produit '1 9 8 8 - 11 - 23'
  faisceau perd  : le 24 juin <annee> 1 9 0 7 -> faisceau a produit '1 9 7 0 - 06 - 24'


### 4.2 Lecture

Le faisceau explore les k chemins les plus vraisemblables **en score
cumulé** : il peut placer en position 2 un jeton un peu moins probable
localement parce que la suite qu'il autorise est globalement meilleure.
Sur cette tâche, la mesure dit autre chose : le glouton et le faisceau
échangent leurs victoires date par date et le solde est nul (−1 point au
modèle final). Les exemples sont instructifs dans les **deux** sens — le
faisceau rattrape des dates où le glouton confond le jour et le mois
(l'émission de `02` après l'année exige de s'être souvenu que `2` était le
jour), et le faisceau perd des dates en transposant des chiffres de
l'année (`1 8 6 2` lu `1 8 2 6`, `1 9 0 7` lu `1 9 7 0`) : garder k
hypothèses ouvertes laisse une quasi-égalité l'emporter en score cumulé
là où le argmax local, plus confiant, avait vu juste. C'est le régime
attendu d'une tâche courte de copie-réordonnancement, où le argmax local
est déjà l'hypothèse dominante : le faisceau paie sur les tâches où une
décision localement plausible engage irréversiblement la suite
(traduction d'une phrase entière), pas sur huit jetons dont quatre sont
copiés.


In [12]:
# --- alignement : ou l'attention croisee lit-elle la source ? -------------
def alignements(modele, paires, n=2):
    """Pas de decodage + poids croises (tetes moyennees, dernier bloc) du DERNIER
    jeton a chaque pas. Retourne (src, jetons emis, poids (pas, B, T_enc))."""
    src, _, _ = encoder_s2s(paires[:n])
    mem = modele.memoire(src)
    poids_capture = []

    def capteur(module, entrees, sortie):
        x, m = entrees
        poids_capture.append(module.poids(x, m).mean(dim=1)[:, -1])  # (B, T_enc)

    crochet = modele.dec_blocs[-1].croisee.register_forward_hook(capteur)
    with torch.no_grad():
        prefixes = torch.full((n, 1), ID_BOS, dtype=torch.long)
        for _ in range(T_DEC):
            logits = modele.pas_decodage(prefixes, mem)
            suiv = logits.argmax(-1)
            prefixes = torch.cat([prefixes, suiv.unsqueeze(1)], dim=1)
            if bool((suiv == ID_EOS).all()):
                break
    crochet.remove()
    return src, prefixes[:, 1:], torch.stack(poids_capture)


src_al, pred_al, poids_al = alignements(s2s, test_dates, n=2)
for b in range(2):
    s, t = test_dates[b]
    print("source :", " ".join(s), "| cible :", " ".join(t))
    jetons = [VOCAB_S2S[j] for j in pred_al[b].tolist() if j not in (ID_EOS, ID_PAD_S)]
    for i, mot in enumerate(jetons):
        if i >= poids_al.shape[0]:
            break
        w = poids_al[i, b]
        top2 = w.topk(2)
        lu = " ; ".join(f"{VOCAB_S2S[src_al[b, k].item()]} ({v:.0%})"
                        for v, k in zip(top2.values, top2.indices))
        print(f"  emis {mot:>4} <- lit surtout : {lu}")
    print()
# ligne de poids complete pour le premier chiffre d'annee emis (exemple 0)
w0 = poids_al[0, 0]
print("poids croises complets (exemple 0, premier jeton emis) :")
for k in range(src_al.shape[1]):
    barre = "#" * int(round(float(w0[k]) * 40))
    print(f"  {VOCAB_S2S[src_al[0, k].item()]:>9} {float(w0[k]):5.0%} {barre}")


source : le 11 novembre <annee> 1 8 3 3 | cible : 1 8 3 3 - 11 - 11
  emis    1 <- lit surtout : 1 (31%) ; le (16%)
  emis    8 <- lit surtout : 8 (24%) ; <annee> (22%)
  emis    3 <- lit surtout : 3 (40%) ; 3 (40%)
  emis    3 <- lit surtout : 3 (37%) ; 3 (37%)
  emis    - <- lit surtout : le (27%) ; 3 (23%)
  emis   11 <- lit surtout : novembre (78%) ; 11 (18%)
  emis    - <- lit surtout : le (49%) ; 11 (24%)
  emis   11 <- lit surtout : 11 (60%) ; novembre (22%)

source : le 2 10 <annee> 1 9 1 0 | cible : 1 9 1 0 - 10 - 02
  emis    1 <- lit surtout : 1 (24%) ; 1 (24%)
  emis    9 <- lit surtout : 9 (26%) ; 2 (19%)
  emis    1 <- lit surtout : 2 (40%) ; 0 (27%)
  emis    0 <- lit surtout : 2 (58%) ; 0 (34%)
  emis    - <- lit surtout : 2 (34%) ; le (16%)
  emis   10 <- lit surtout : 10 (68%) ; 1 (6%)
  emis    - <- lit surtout : le (34%) ; <annee> (19%)
  emis   10 <- lit surtout : <annee> (26%) ; 10 (24%)

poids croises complets (exemple 0, premier jeton emis) :
         le   16% #

### 4.3 Lecture

L'alignement appris se lit directement : les chiffres de l'année émis en
début de cible pointent vers les **jetons d'année en fin de source**, le
mois vers le **jeton de mois**, le jour vers le **jeton de jour** — alors
qu'aucune supervision d'alignement n'a été donnée ; seule l'exact-match sur
la date complète a supervisé le modèle. L'attention croisée a dû construire
cette correspondance pour transporter la bonne information au bon pas.
C'est la démonstration du rôle que l'article de 2017 donnait à l'attention
dans la traduction, reproduite sur une tâche où chaque position reste
vérifiable à la main.


## Exercice 2 — un pas du faisceau, à la main

**Objectif.** Écrire `meilleurs_successeurs(scores, logp, finis)` : étant
donné les scores cumulés `(k,)` des candidats du faisceau, les log-probs
`(k, V)` du jeton suivant pour chaque candidat et le masque booléen `(k,)`
des candidats déjà finis, retourner les indices `(k,)` des k meilleures
paires (candidat, jeton) dans le tableau aplati `k*V` — en interdisant le
jeton `<pad>` (colonne 0 de l'exemple) et en laissant les candidats finis
concourir uniquement via leur jeton `<eos>` (colonne 2 de l'exemple, à
coût nul).


In [13]:
def meilleurs_successeurs(scores, logp, finis):
    """Exercice 2 : indices (k,) des k meilleures paires (candidat, jeton) aplaties."""
    # TODO etudiant : cumuler scores + logp, masquer pad et les non-eos des finis, topk sur l'aplat
    resultat = None
    return resultat


# verification (a completer apres l'exercice) :
k_ex, V_ex = 3, 6  # colonnes : 0 = <pad>, 2 = <eos> (exemple abstrait)
scores_ex = torch.tensor([0.0, -1.0, -2.0])
logp_ex = torch.log_softmax(torch.tensor([[2.0, 1.0, 0.5, 0.0, 0.0, 0.0],
                                          [0.0, 0.0, 0.0, 0.0, 3.0, 0.0],
                                          [4.0, 0.0, 0.0, 0.0, 0.0, 0.0]]), dim=-1)
finis_ex = torch.tensor([False, False, True])
idx = meilleurs_successeurs(scores_ex, logp_ex, finis_ex)
if idx is None:
    print("Exercice 2 a completer : meilleurs_successeurs a renvoye None")
else:
    print("indices retenus :", idx.tolist(), "| (candidat, jeton) =",
          [(int(c), int(j)) for c, j in zip(idx // V_ex, idx % V_ex)])
    print("attendu : le candidat 2 (fini) ne concourt que par la colonne 2 (<eos>), la colonne 0 (<pad>) est exclue")


Exercice 2 a completer : meilleurs_successeurs a renvoye None


## 5. Les trois familles, côte à côte

Les trois architectures partagent les mêmes briques d'attention (organe
`tv`) — seuls la **forme du modèle** et le **masque** changent. Le tableau
suivant récapitule ce que chaque famille peut et ne peut pas faire, avec les
mesures de ce carnet en regard du décodeur seul déjà couvert par la série.


In [14]:
ref_lm = PetitLM(vocab=64, d_model=D_MODEL, n_heads=N_TETES, n_kv_heads=N_TETES,
                 window=None, n_couches=N_COUCHES)
print(f"parametres comptes (d_model={D_MODEL}, {N_TETES} tetes, {N_COUCHES} couches, vocab 64) :")
print(f"  decodeur seul      (PetitLM)          : {sum(p.numel() for p in ref_lm.parameters()):>9,}")
print(f"  encodeur seul      (PetitEncodeur)    : {enc_bi.n_parametres():>9,}")
print(f"  encodeur-decodeur  (EncodeurDecodeur) : {s2s.n_parametres():>9,}")


parametres comptes (d_model=64, 4 tetes, 2 couches, vocab 64) :
  decodeur seul      (PetitLM)          :   107,776
  encodeur seul      (PetitEncodeur)    :   103,424
  encodeur-decodeur  (EncodeurDecodeur) :   239,744


### 5.1 Lecture

Trois lectures du tableau :

- **Le compte de paramètres** suit la structure, pas la tâche : un
  encodeur-décodeur à dimensions égales coûte environ deux fois un modèle
  unilatéral, parce que chaque couche de décodeur porte **trois**
  sous-blocs (self-attention, attention croisée, MLP) contre deux.
- **« Peut générer » n'est pas un détail** : c'est une propriété du masque.
  L'encodeur de la section 3 atteint sa précision précisément parce qu'il
  lit le contexte droit — le même modèle ne peut plus générer un jeton sans
  re-encoder toute la phrase.
- **Le mauvais outil se paie** : le témoin 3.4 (causal sur MLM) et la
  section 4 (l'exact-match s'effondre si l'année n'est pas lue en source)
  sont les deux faces du même principe — la famille se choisit par la
  tâche, pas par la mode.


## Exercice 3 — le masque de remplissage de la source

**Objectif.** En production, les sources d'un lot n'ont pas toutes la même
longueur : on les pad à droite, et l'attention croisée ne doit pas lire les
positions de remplissage. Écrire `masque_remplissage(longueurs, T_enc)` :
le booléen `(B, T_enc)` vrai aux positions **valides** — la position `j` est
valide pour l'exemple `b` si `j < longueurs[b]`.


In [15]:
def masque_remplissage(longueurs, T_enc):
    """Exercice 3 : (B, T_enc) vrai aux positions valides, faux sur le pad."""
    # TODO etudiant : un arange diffuse contre un view, une comparaison
    resultat = None
    return resultat


# verification (a completer apres l'exercice) :
long_ex = torch.tensor([8, 5, 3])
m_ex = masque_remplissage(long_ex, T_enc=8)
if m_ex is None:
    print("Exercice 3 a completer : masque_remplissage a renvoye None")
else:
    print("forme :", tuple(m_ex.shape))
    for b in range(3):
        attendu = torch.arange(8) < long_ex[b]
        print(f"  ligne {b} (longueur {long_ex[b].item()}) :",
              int(m_ex[b].sum()), "valides | conforme :", bool((m_ex[b] == attendu).all()))


Exercice 3 a completer : masque_remplissage a renvoye None


## 6. Conclusion

**Ce qu'on a mesuré, et ce que chaque chiffre autorise à conclure.**

| Affirmation | Preuve dans ce notebook | Verdict |
|---|---|---|
| `attn_pleine` et `AttnCroisee` implémentent la forme softmax(QK^T/√d)V | vérifications (a), (b) : écart max ~1e-7 contre boucle par tête | etabli |
| Le bloc bidirectionnel diffère du causal à poids égaux | selfcheck (c), écart non nul dès la position 0 | etabli |
| L'organe TV-00b n'a pas régressé | selfcheck (d) : bande contre masque, inchangé | etabli |
| La bidirectionnalité sert le MLM, là où la preuve est à droite du trou — et coûte là où elle est à gauche | sec. 3.4 : sujet 63.7 % (bi) contre 31.6 % (causal), contrôle symétrique verbe 41.9 % contre 90.0 % | etabli sur ce corpus |
| Le taux de masquage dégrade la précision quand il monte | sec. 3.3 : précision à 5 / 15 / 30 %, les deux positions sondées décroissent | etabli sur ce corpus |
| L'attention croisée apprend l'alignement sans supervision d'alignement | sec. 4.3 : année émise lit l'année source, mois lit mois, jour lit jour | etabli |
| Le faisceau rattrape des dates que le glouton rate — et en perd autant | sec. 4.2 : victoires et défaites date par date, solde -1 point au modèle final | mesuré sur ce test |

**Ce que ce carnet ne mesure pas.** Les plafonds (taille du vocabulaire,
longueur des séquences, une seule attention croisée par bloc) sont ceux
d'un jouet CPU lisible, pas d'un banc : les valeurs numériques ne se
généralisent pas au-delà du micro-langage et des 500 dates de test. Le
pendant industriel de ces deux familles (un vrai encodeur et un vrai
seq2seq chargés via `transformers`) reste ouvert côté bloc B. Les valeurs numériques sont de plus propres au couple torch épinglé (2.13.0+cu126) et à la graine : rejoué sous un autre bâtiment de torch, des cellules individuelles bougent — jusqu’à plusieurs points sur les colonnes fines, notamment le témoin causal ré-entraîné — alors que les classements directionnels tiennent.


***
Références (paraphrasées, aucun texte recopié) :

- Vaswani et al., *Attention Is All You Need*, 2017 (arXiv 1706.03762) —
  l'architecture encodeur-décodeur et l'attention croisée.
- Devlin et al., *BERT: Pre-training of Deep Bidirectional Transformers*,
  2018 (arXiv 1810.04805) — le MLM bidirectionnel et le masquage 80/10/10.
- Raffel et al., *Exploring the Limits of Transfer Learning with a Unified
  Text-to-Text Transformer*, 2019 (arXiv 1910.10683) — la famille
  encodeur-décodeur appliquée à tout.

Retour : [README de la série](README.md) · notebook précédent (bloc A) :
[TV-00b — Variantes d'attention](TV-00b-Attention-Variants-from-scratch.ipynb) ·
le décodeur seul : `3.4-Attention-Transformer-From-Scratch` (série 03-DeepLearning).
